# Tune Paper candidate windows

Choose Paper's window sizes and stride cap for each projection shape.
Use the measured profile from [01](../01/01_01_measure_chunk_latency.ipynb)
and projection shapes from [02](../02/02_01_capture_traces.ipynb).

Run from this directory on the target device with CUDA and the upstream native
extension. [03_02](03_02_measure_latency.ipynb) uses the selected settings.

In [14]:
from pathlib import Path
from time    import perf_counter_ns

import json
import os
import sys
import torch

import numpy  as np
import pandas as pd

In [15]:
UPSTREAM = Path("../../vlm-flash")

sys.path.insert(0, str((UPSTREAM / "src").resolve()))
sys.path.insert(0, str((UPSTREAM / "scripts").resolve()))

import vlmflash
from vlmflash._native import native
from _lock            import exclusive

## 1. Inputs

Use the median cost at each profiled size. Tuning uses random importance
with seed 0, sparsity 0.1, and one Torch CPU thread.

In [16]:
PROFILE = Path("../01/chunk_latency_measurements.csv")
TRACE   = Path("../02/activation_traces.json")

In [17]:
os.environ.setdefault("MAX_JOBS", "1")
torch.set_num_threads(1)

assert torch.cuda.is_available(), "Paper tuning requires CUDA."
assert native() is not None, "Build the upstream native extension first."

In [18]:
archive = json.loads(TRACE.read_text())

weight_bytes = archive["weight_bytes"]
shapes       = sorted({(trace["n"], trace["d"]) for trace in archive["traces"]})

In [19]:
measurements = pd.read_csv(PROFILE)

sizes = np.sort(measurements["size_kib"].unique()) # KiB

latency_samples = (
    measurements.pivot(
        index   = "size_kib",
        columns = "repeat",
        values  = "latency_ms",
    )
    .sort_index()
    .sort_index(axis=1)
    .to_numpy(float)
)

latencies = np.median(latency_samples, axis=1)

latency_table = vlmflash.LatencyTable(dict(zip(sizes, latencies)))

## 2. Candidate times

Sweep window starts and stride caps from 4 to 64 KiB in 4 KiB steps.
The start also sets the window-size increment; the profile sets the upper bound.
Evaluate each distinct row-quantized configuration once per shape.

After one warmup, take the median of 30 calls. Timing includes wrapper preparation
and GPU sorting; 03_02 prepares candidates before timing.

In [20]:
REPEATS  = 30
SPARSITY = 0.1

In [21]:
def tuning_trials(n, d):
    # Importance and budget
    generator = torch.Generator().manual_seed(0)
    values    = torch.rand(n, generator=generator) + 0.5
    budget    = n - int(n * SPARSITY)

    # Candidate windows
    row_kib = d * weight_bytes / 1024
    seen    = set()

    for start in range(4, 68, 4):
        for jump in range(4, 68, 4):

        # 1. ROW GEOMETRY
            start_rows = max(1, int(start / row_kib))
            jump_rows  = max(1, int(jump / row_kib))
            geometry   = (start_rows, jump_rows)

            # Skip KiB settings that map to the same row counts.
            if geometry in seen:
                continue
            seen.add(geometry)

        # 2. SELECTOR
            params = vlmflash.ChunkParams(
                start_kb    = start,
                jump_cap_kb = jump,
            )

            def select():
                return vlmflash.select_chunks(
                    values, 
                    budget, 
                    row_kib, 
                    latency_table, 
                    params,
                    impl = "native",
                )

        # 3. WARMUP
            try:
                select()
            except ValueError:
                continue

        # 4. TIMING
            times = []

            for _ in range(REPEATS):
                begin = perf_counter_ns()
                select()
                times.append((perf_counter_ns() - begin) / 1e6)

            overhead = round(float(np.median(times)), 3)

        # 5. RESULT
            yield {
                "n"           : n,
                "d"           : d,
                "start_kb"    : start,
                "jump_cap_kb" : jump,
                "overhead_ms" : overhead,
            }

In [22]:
records = []

with exclusive("experiment03"):
    for n, d in shapes:
        records.extend(tuning_trials(n, d))
        print(f"Tuned {n}x{d}", flush=True)

trials = pd.DataFrame(records)

Tuned 896x128
Tuned 896x896
Tuned 896x4864
Tuned 4864x896


In [23]:
trials.head()

,n,d,start_kb,jump_cap_kb,overhead_ms
0,896,128,4,4,0.138
1,896,128,4,8,0.129
2,896,128,4,12,0.121
3,896,128,4,16,0.117
4,896,128,4,20,0.114


## 3. Selected settings

Use candidates within 1.6 ms, falling back to those within 2 ms. Minimize start
plus stride cap, breaking ties by smaller start. If none meets 2 ms, use the
fastest candidate. Apply this rule to medians rounded to 0.001 ms.

In [24]:
LIMIT_MS     = 2.0
PREFERRED_MS = 1.6

In [25]:
selected = []

for (n, d), group in trials.groupby(["n", "d"], sort=True):

# 1. FEASIBLE CONFIGURATIONS
    feasible = group[group.overhead_ms <= LIMIT_MS]

# 2. SAFETY MARGIN
    preferred = feasible[feasible.overhead_ms <= PREFERRED_MS]
    pool      = preferred if not preferred.empty else feasible

# 3. RANKING
    if pool.empty:
        # Fall back to the fastest configuration.
        ranked = group.sort_values("overhead_ms", kind="stable")
    else:
        # Prefer smaller windows and stride caps.
        ranked = pool.assign(geometry_sum=pool.start_kb + pool.jump_cap_kb)
        ranked = ranked.sort_values(
            ["geometry_sum", "start_kb"],
            kind = "stable",
        )

    best = ranked.iloc[0]

# 4. RESULT
    selected.append({
        "n"             : int(n),
        "d"             : int(d),
        "start_kb"      : int(best.start_kb),
        "jump_cap_kb"   : int(best.jump_cap_kb),
        "overhead_ms"   : best.overhead_ms,
        "num_feasible"  : len(feasible),
        "num_evaluated" : len(group),
    })

tuning = pd.DataFrame(selected)
tuning

,n,d,start_kb,jump_cap_kb,overhead_ms,num_feasible,num_evaluated
0,896,128,4,4,0.138,256,256
1,896,896,4,4,1.217,256,256
2,896,4864,4,4,1.184,36,36
3,4864,896,8,16,0.917,247,256


## 4. Output

Save candidate times to `paper_tuning_trials.csv` and selected settings to
`paper_tuning.csv`. Run 03_02 to measure with the new settings.

Historical settings were recovered from the original measurement environment;
candidate timings were not retained. Existing measurements keep their tuning
snapshot in `environment.json`.

In [26]:
trials.to_csv("paper_tuning_trials.csv", index=False)
tuning.to_csv("paper_tuning.csv", index=False)